In [3]:
#import libraries

import pandas as pd
import numpy as np
import sqlite3
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

print("✓ Libraries imported")

✓ Libraries imported


In [4]:
# Define reference data

# ── Subscription plans ─────────────────────────────────────
plans = [
    {"plan": "Basic",      "monthly_price": 29,  "weight": 40},
    {"plan": "Pro",        "monthly_price": 79,  "weight": 35},
    {"plan": "Business",   "monthly_price": 149, "weight": 20},
    {"plan": "Enterprise", "monthly_price": 399, "weight": 5},
]

# ── Acquisition channels ───────────────────────────────────
channels = ["Organic Search", "Paid Ads", "Referral", "Social Media", "Direct"]

# ── Churn rates by plan (monthly probability of churning) ──
churn_rates = {
    "Basic":      0.08,   # 8% monthly churn
    "Pro":        0.05,   # 5% monthly churn
    "Business":   0.03,   # 3% monthly churn
    "Enterprise": 0.02,   # 2% monthly churn
}

print("✓ Reference data defined")
print(f"  Plans: {[p['plan'] for p in plans]}")
print(f"  Channels: {channels}")

✓ Reference data defined
  Plans: ['Basic', 'Pro', 'Business', 'Enterprise']
  Channels: ['Organic Search', 'Paid Ads', 'Referral', 'Social Media', 'Direct']


In [5]:
#Generate subscribers

# ── Generate subscriber signups ────────────────────────────
# 2,000 subscribers signing up between Jan 2022 and Dec 2023
# (leaves 2024 as the observation window for retention)

n_subscribers = 2000
start_date = datetime(2022, 1, 1)
end_date   = datetime(2023, 12, 31)
days_range = (end_date - start_date).days

subscribers = []
for i in range(n_subscribers):
    # Pick a plan weighted by popularity
    plan_choice = random.choices(
        plans, weights=[p["weight"] for p in plans])[0]

    # Signup date with slight growth trend — more signups in later months
    day_offset  = int(np.random.triangular(0, days_range, days_range))
    signup_date = start_date + timedelta(days=day_offset)

    subscribers.append({
        "subscriber_id": f"SUB{str(i+1).zfill(4)}",
        "signup_date":   signup_date.strftime("%Y-%m-%d"),
        "plan":          plan_choice["plan"],
        "monthly_price": plan_choice["monthly_price"],
        "channel":       random.choice(channels),
    })

subscribers_df = pd.DataFrame(subscribers)
subscribers_df["signup_date"] = pd.to_datetime(subscribers_df["signup_date"])
subscribers_df["cohort_month"] = subscribers_df["signup_date"].dt.to_period("M")

print(f"✓ {len(subscribers_df):,} subscribers generated")
print(f"✓ Date range: {subscribers_df['signup_date'].min().date()} "
      f"to {subscribers_df['signup_date'].max().date()}")
print(f"\nPlan breakdown:")
print(subscribers_df["plan"].value_counts())

✓ 2,000 subscribers generated
✓ Date range: 2022-02-11 to 2023-12-30

Plan breakdown:
plan
Basic         814
Pro           707
Business      371
Enterprise    108
Name: count, dtype: int64


In [6]:
#Generate monthly activity

# ── Generate monthly activity records ──────────────────────
# For each subscriber, simulate whether they are active
# each month from signup until they churn or Dec 2024

observation_end = datetime(2024, 12, 31)
activity_records = []

for _, sub in subscribers_df.iterrows():
    signup_dt   = sub["signup_date"].to_pydatetime()
    plan        = sub["plan"]
    churn_rate  = churn_rates[plan]
    current_dt  = signup_dt.replace(day=1)
    month_num   = 0

    while current_dt <= observation_end:
        # First month — always active (month 0)
        if month_num == 0:
            is_active = True
        else:
            # Each month has a chance of churning
            # Early months have slightly higher churn
            early_churn_boost = 1.3 if month_num <= 3 else 1.0
            adjusted_churn    = min(churn_rate * early_churn_boost, 1.0)
            is_active         = random.random() > adjusted_churn

        if not is_active:
            break  # Once churned, stop generating records

        activity_records.append({
            "subscriber_id": sub["subscriber_id"],
            "plan":          plan,
            "monthly_price": sub["monthly_price"],
            "channel":       sub["channel"],
            "cohort_month":  str(sub["cohort_month"]),
            "activity_month": current_dt.strftime("%Y-%m"),
            "month_number":  month_num,
            "is_active":     1,
        })

        # Move to next month
        if current_dt.month == 12:
            current_dt = current_dt.replace(year=current_dt.year + 1, month=1)
        else:
            current_dt = current_dt.replace(month=current_dt.month + 1)
        month_num += 1

activity_df = pd.DataFrame(activity_records)

print(f"✓ {len(activity_df):,} monthly activity records generated")
print(f"✓ Unique subscribers: {activity_df['subscriber_id'].nunique():,}")
print(f"✓ Avg months active per subscriber: "
      f"{len(activity_df) / activity_df['subscriber_id'].nunique():.1f}")

✓ 23,650 monthly activity records generated
✓ Unique subscribers: 2,000
✓ Avg months active per subscriber: 11.8


In [8]:
#Save to SQLite and CSV

# ── Save to SQLite ─────────────────────────────────────────
conn = sqlite3.connect("../data/cohort.db")

subscribers_df["cohort_month"] = subscribers_df["cohort_month"].astype(str)
subscribers_df.to_sql("subscribers", conn, if_exists="replace", index=False)
activity_df.to_sql("activity",    conn, if_exists="replace", index=False)

conn.close()
print("✓ Database saved to data/cohort.db")

# ── Save to CSV ────────────────────────────────────────────
subscribers_df.to_csv("../data/subscribers.csv", index=False)
activity_df.to_csv("../data/activity.csv",    index=False)
print("✓ CSVs saved to data/")

# ── Sanity check ───────────────────────────────────────────
conn = sqlite3.connect("../data/cohort.db")
print("\n── Tables in database ──────────────────────────────")
print(pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
).to_string(index=False))

print("\n── Subscribers per cohort month (first 6) ──────────")
print(pd.read_sql("""
    SELECT cohort_month,
           COUNT(*)                     AS subscribers,
           COUNT(DISTINCT plan)         AS plans
    FROM subscribers
    GROUP BY cohort_month
    ORDER BY cohort_month
    LIMIT 6
""", conn).to_string(index=False))
conn.close()

✓ Database saved to data/cohort.db
✓ CSVs saved to data/

── Tables in database ──────────────────────────────
       name
subscribers
   activity

── Subscribers per cohort month (first 6) ──────────
cohort_month  subscribers  plans
     2022-02            8      3
     2022-03           17      4
     2022-04           31      3
     2022-05           37      4
     2022-06           33      4
     2022-07           54      4
